# Exploration MVTec AD — catégorie `bottle`
Change `CATEGORY` ci-dessous pour explorer une autre catégorie.

In [ ]:
from pathlib import Path
from collections import Counter

from PIL import Image
import matplotlib.pyplot as plt

CATEGORY = "bottle"
ROOT = Path("../data/mvtec") / CATEGORY
assert ROOT.exists(), f"{ROOT} introuvable — lance d'abord: python -m src.data.download --category {CATEGORY}"

## 1. Comptage d'images par split et par type de défaut

In [ ]:
counts = Counter()
for split_dir in (ROOT / "train").iterdir():
    counts[f"train/{split_dir.name}"] = len(list(split_dir.glob("*.png")))
for split_dir in (ROOT / "test").iterdir():
    counts[f"test/{split_dir.name}"] = len(list(split_dir.glob("*.png")))
for name, n in sorted(counts.items()):
    print(f"{name:30s} {n}")

## 2. Exemples : image normale vs défectueuse + masque

In [ ]:
defect_types = [d.name for d in (ROOT / "test").iterdir() if d.name != "good"]
defect_type = defect_types[0]

good_img = next((ROOT / "test" / "good").glob("*.png"))
defect_img = next((ROOT / "test" / defect_type).glob("*.png"))
mask_img = ROOT / "ground_truth" / defect_type / f"{defect_img.stem}_mask.png"

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(Image.open(good_img)); axes[0].set_title("normale")
axes[1].imshow(Image.open(defect_img)); axes[1].set_title(f"défaut: {defect_type}")
axes[2].imshow(Image.open(mask_img), cmap="gray"); axes[2].set_title("masque ground truth")
for ax in axes: ax.axis("off")
plt.show()

## 3. Distribution des tailles d'image

In [ ]:
sizes = Counter()
for img_path in ROOT.rglob("*.png"):
    with Image.open(img_path) as img:
        sizes[img.size] += 1
print(sizes)

## 4. Sanity checks : doublons et extensions

In [ ]:
import hashlib

extensions = Counter(p.suffix for p in ROOT.rglob("*") if p.is_file())
print("Extensions:", extensions)

hashes = Counter()
for img_path in ROOT.rglob("*.png"):
    hashes[hashlib.md5(img_path.read_bytes()).hexdigest()] += 1
duplicates = {h: n for h, n in hashes.items() if n > 1}
print(f"Doublons détectés: {len(duplicates)}")